# 20. Ekspor Core ML — View Validator (5 kelas + OOD)

**Prasyarat — jalankan dulu sebelum notebook ini:**
1. Notebook 17 (retrain) — pastikan `data/view5` sudah pakai folder maxillary/mandibular
   yang sudah dibetulkan (bukan yang lama/tertukar).
2. Notebook 18 (rebuild OOD) — supaya `models/view_ood_config.json` juga fresh dari model
   yang baru, bukan config lama.

Notebook ini murni packaging: load `view5_classifier.pt` + `view_ood_config.json`, bungkus
jadi satu model Core ML 2-keluaran (skor 5-kelas + vektor fitur untuk OOD), plus config.json
untuk dibaca Swift — pola identik dengan ekspor `ACGrader` di notebook 10 bagian 12, yang
sudah terbukti jalan di app (pesan "Photo is out of the model's scope" yang kamu lihat itu
asalnya dari situ).

In [ ]:
import os, copy, json
import numpy as np
import torch, torch.nn as nn
from torchvision import models

PROJECT_ROOT = os.path.expanduser('~/IOTN-AC')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'models')
VIEW_CLASSES_5 = ['frontal', 'lateral_kanan', 'lateral_kiri', 'maxillary', 'mandibular']

_ckpt5 = os.path.join(MODELS_DIR, 'view5_classifier.pt')
_cfg_path = os.path.join(MODELS_DIR, 'view_ood_config.json')
assert os.path.exists(_ckpt5), f'Model belum ada: {_ckpt5} — jalankan notebook 17 dulu.'
assert os.path.exists(_cfg_path), f'Config OOD belum ada: {_cfg_path} — jalankan notebook 18 dulu.'

with open(_cfg_path) as f:
    _cfg = json.load(f)
assert _cfg['view_classes'] == VIEW_CLASSES_5, (
    'Urutan kelas di view_ood_config.json tidak sama dengan VIEW_CLASSES_5 — cek lagi notebook 18.')
print('Config OOD dimuat: k_centroid =', _cfg['k_centroid'], '| threshold =', _cfg['ood_threshold'])

## 1. Muat model, pisah backbone + kepala (sama seperti notebook 18)

In [ ]:
def buat_resnet_cls(num_classes=len(VIEW_CLASSES_5), dropout=0.3):
    m = models.resnet18(weights=None)
    m.fc = nn.Sequential(nn.Dropout(dropout), nn.Linear(m.fc.in_features, num_classes))
    return m

def pisah_resnet(m):
    mm = copy.deepcopy(m).cpu().eval(); kepala = mm.fc; mm.fc = nn.Identity(); return mm, kepala

_model5 = buat_resnet_cls()
_model5.load_state_dict(torch.load(_ckpt5, map_location='cpu'))
TULANG, KEPALA = pisah_resnet(_model5)
print('Backbone + kepala siap dibungkus untuk ekspor.')

## 2. Bungkus jadi satu modul (normalisasi ImageNet tertanam) + trace + convert

In [ ]:
RATA, SIMPANG = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

class ViewValidatorCoreML(nn.Module):
    """Masukan: RGB 1x3x224x224 dalam [0,1]. Keluaran: (skor 5-kelas, vektor fitur norm-1).
    Normalisasi ImageNet ditanam, sama seperti ACGrader — app cukup kirim piksel mentah."""
    def __init__(self, tulang, kepala, rata, simpang):
        super().__init__()
        self.tulang = tulang.eval(); self.kepala = kepala.eval()
        self.register_buffer('rata', torch.tensor(rata).view(1, 3, 1, 1))
        self.register_buffer('simpang', torch.tensor(simpang).view(1, 3, 1, 1))

    def forward(self, x):
        x = (x - self.rata) / self.simpang
        f = self.tulang(x)
        skor = self.kepala(f)
        vektor = f / (f.norm(dim=1, keepdim=True) + 1e-9)
        return skor, vektor

paket = ViewValidatorCoreML(copy.deepcopy(TULANG), copy.deepcopy(KEPALA), RATA, SIMPANG).eval()
contoh = torch.rand(1, 3, 224, 224)
with torch.no_grad():
    terlacak = torch.jit.trace(paket, contoh)
print('Traced OK — output shapes:', [tuple(o.shape) for o in paket(contoh)])

In [ ]:
try:
    import coremltools as ct
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'coremltools']); import coremltools as ct

mlmodel = ct.convert(
    terlacak,
    inputs=[ct.ImageType(name='image', shape=(1, 3, 224, 224), scale=1/255.0, color_layout=ct.colorlayout.RGB)],
    outputs=[ct.TensorType(name='scores'), ct.TensorType(name='vektor')],
    minimum_deployment_target=ct.target.iOS15)
mlmodel.short_description = ('View validator (5 kelas: frontal/lateral_kanan/lateral_kiri/'
                              'maxillary/mandibular) + vektor fitur untuk OOD.')

_mlpath = os.path.join(MODELS_DIR, 'ViewValidator.mlpackage')
mlmodel.save(_mlpath)
print('Core ML tersimpan ->', os.path.relpath(_mlpath, PROJECT_ROOT))

## 3. Simpan config untuk Swift (nama field cocok dengan `ViewValidatorRegressor.swift`)

In [ ]:
cfg_swift = dict(
    view_classes=VIEW_CLASSES_5,          # urutan HARUS sama dengan ViewValidatorRegressor.classOrder
    ood_threshold=round(float(_cfg['ood_threshold']), 6),
    centroid=_cfg['centroid'],
)
_cfgpath = os.path.join(MODELS_DIR, 'ViewValidator_config.json')
with open(_cfgpath, 'w') as f:
    json.dump(cfg_swift, f, indent=1)
print('Config tersimpan ->', os.path.relpath(_cfgpath, PROJECT_ROOT))
print('\nSalin KEDUANYA ke Malokit/MLModels/ di repo iOS:')
print('  - models/ViewValidator.mlpackage')
print('  - models/ViewValidator_config.json')

## 4. Sanity check — bandingkan skor PyTorch vs Core ML pada satu foto

In [ ]:
from PIL import Image
from glob import glob

_contoh_path = glob(os.path.join(PROJECT_ROOT, 'data', 'view5', 'test', 'frontal', '*.[Jj][Pp][Gg]'))[0]
_img = Image.open(_contoh_path).convert('RGB').resize((224, 224))
_x = torch.from_numpy(np.asarray(_img).transpose(2, 0, 1) / 255.0).float().unsqueeze(0)
with torch.no_grad():
    _skor_pt, _vek_pt = paket(_x)

try:
    _hasil = mlmodel.predict({'image': _img})
    _skor_cm = np.array(_hasil['scores']); _vek_cm = np.array(_hasil['vektor'])
    print('Skor PyTorch:', np.round(_skor_pt.numpy().ravel(), 3))
    print('Skor Core ML:', np.round(_skor_cm.ravel(), 3))
    print('Selisih maks (skor):', float(np.abs(_skor_pt.numpy().ravel() - _skor_cm.ravel()).max()), '(harus ~0)')
    print('Selisih maks (vektor):', float(np.abs(_vek_pt.numpy().ravel() - _vek_cm.ravel()).max()), '(harus ~0)')
except Exception as e:
    print('Prediksi Core ML hanya jalan di macOS:', type(e).__name__)

## Langkah selanjutnya

1. Salin `models/ViewValidator.mlpackage` dan `models/ViewValidator_config.json` ke
   `Malokit/MLModels/` di repo `malokit-app` (branch `feature/validator-input`).
2. Tambahkan keduanya ke target Xcode `Malokit` (drag ke project navigator, centang
   "Add to targets: Malokit").
3. `ViewValidatorRegressor.swift` + integrasi di `CaptureFlowView.swift`/`ReviewView.swift`
   sudah disiapkan terpisah — begitu file model ada di bundle, langsung aktif.